# CA Exam Checker AI — Evaluation Supervisor Fine-Tuning

This notebook trains a **separate research adapter** to audit evaluation integrity. The policy covers the student's selected checking mode (`standard`, `strict`, or `lenient`), valid alternative methods, proportional partial credit, and consequential marking. It does not replace the primary examiner or overwrite the existing study-planner model.

**Safety notes**
- The initial train/validation/test sets are synthetic; no student answer sheets or ICAI solution text are included.
- Upload the matching MTP/PYQ question paper and Suggested Answer in the source-pack section. Those PDFs are kept in your Drive as paired references, not treated as labelled training examples without verified student-answer scores.
- A synthetic test score is only a pipeline smoke test, not proof that marks are academically correct.
- Keep the adapter advisory. Deterministic integrity gates and human review remain authoritative when evidence or references are uncertain.


In [ ]:
import os, sys, subprocess, json, torch
from pathlib import Path

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU runtime required. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun this cell."
    )
print("✅ GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
REPO_DIR = "/content/AI-Exam"
BRANCH = "main"
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "https://github.com/thomasshelby-pixel/AI-Exam.git", REPO_DIR],
        check=True
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

PROJECT_DIR = f"{REPO_DIR}/research/ca-evaluation-supervisor-ai"
%cd /content/AI-Exam/research/ca-evaluation-supervisor-ai
%pip install -q -r requirements-colab.txt
print("✅ Training sources ready:", PROJECT_DIR)


## Upload matching MTP/PYQ reference files

Upload the **question paper** and its matching **Suggested Answer**. Add the official marking scheme/rubric too when available. These stay in your Google Drive as a paired, versioned source pack; they are not uploaded to GitHub.

**Important:** reference PDFs alone are not labelled fine-tuning examples. We must also have student-answer examples with human-verified marks/reasons before using real material for supervised training. The starter training dataset remains synthetic so it cannot silently learn incorrect scores from an unlabelled PDF.


In [ ]:
import fitz, hashlib, re
from datetime import datetime, timezone
from google.colab import files

def slugify(value):
    value = re.sub(r'[^A-Za-z0-9._-]+', '_', (value or '').strip())
    return value.strip('._-')[:80] or 'MTP_PYQ_reference'

source_label = input('Source label (e.g. CA Inter Nov 2025 Taxation MTP 1): ').strip()
if not source_label:
    raise ValueError('Please give this MTP/PYQ source pack a label.')
pack_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
REFERENCE_DIR = Path('/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_reference_materials') / f'{slugify(source_label)}_{pack_stamp}'
REFERENCE_DIR.mkdir(parents=True, exist_ok=False)

def upload_pdf(role, required=True):
    print(f'Upload exactly one PDF for: {role}. Cancel/skip only if this file is optional.')
    uploaded = files.upload()
    if not uploaded:
        if required:
            raise ValueError(f'{role} is required.')
        return None
    if len(uploaded) != 1:
        raise ValueError(f'Upload one PDF at a time for {role}.')
    original_name, raw_bytes = next(iter(uploaded.items()))
    if not original_name.lower().endswith('.pdf'):
        raise ValueError(f'{role} must be a PDF: {original_name}')
    pdf = fitz.open(stream=raw_bytes, filetype='pdf')
    extracted = '\n\n'.join(f'--- Page {page.number + 1} ---\n{page.get_text("text")}' for page in pdf)
    extracted = extracted.strip()
    if len(extracted) < 100:
        raise ValueError(f'{original_name} appears scanned/image-only or has too little selectable text. Provide a text-searchable PDF or OCR it first; do not silently train on empty text.')
    safe_role = slugify(role.lower())
    pdf_path = REFERENCE_DIR / f'{safe_role}.pdf'
    text_path = REFERENCE_DIR / f'{safe_role}.txt'
    pdf_path.write_bytes(raw_bytes)
    text_path.write_text(extracted, encoding='utf-8')
    return {
        'role': role,
        'original_filename': original_name,
        'stored_pdf': str(pdf_path),
        'extracted_text': str(text_path),
        'sha256': hashlib.sha256(raw_bytes).hexdigest(),
        'page_count': len(pdf),
        'extracted_characters': len(extracted),
    }

sources = []
sources.append(upload_pdf('question_paper', required=True))
sources.append(upload_pdf('suggested_answer', required=True))
marking_scheme = upload_pdf('marking_scheme', required=False)
if marking_scheme:
    sources.append(marking_scheme)

manifest = {
    'source_label': source_label,
    'source_type': 'MTP/PYQ',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'files': sources,
    'storage_scope': 'Google Drive supplied by the user; not committed to the repository',
    'training_status': 'REFERENCE_ONLY_UNTIL_HUMAN_VERIFIED_STUDENT_ANSWER_LABELS_EXIST',
}
(REFERENCE_DIR / 'reference_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('✅ Paired reference pack saved:', REFERENCE_DIR)
print(json.dumps(manifest, indent=2))


In [ ]:
# Generate reproducible synthetic examples for train / validation / held-out test.
# The generator does not read Google Drive, student uploads, or ICAI PDFs.
subprocess.run(
    [sys.executable, "generate_supervisor_dataset.py",
     "--output-dir", "/content/ca-evaluation-supervisor-data",
     "--train-per-category", "160",
     "--validation-per-category", "30",
     "--test-per-category", "40",
     "--seed", "20261017"],
    check=True
)
manifest = json.loads(Path("/content/ca-evaluation-supervisor-data/dataset_manifest.json").read_text())
print(json.dumps(manifest, indent=2))
assert manifest["dataset_kind"] == "synthetic_only"
assert manifest["counts"]["train"]["examples"] > manifest["counts"]["test"]["examples"]


## Train an isolated QLoRA adapter

The trainer deliberately refuses CPU-only runtime and refuses output locations that could overwrite the old planner model. Default training uses a T4 GPU, 4-bit QLoRA, and a new versioned Drive output folder.

Do not rerun the training cell while it is running. A completed run should save adapter weights plus a training manifest.


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_smollm2_360m_v1_20261017"
subprocess.run(
    [sys.executable, "train_supervisor_qlora.py",
     "--train-file", "/content/ca-evaluation-supervisor-data/train.jsonl",
     "--validation-file", "/content/ca-evaluation-supervisor-data/validation.jsonl",
     "--output-dir", OUTPUT_DIR,
     "--base-model", "HuggingFaceTB/SmolLM2-360M-Instruct",
     "--epochs", "3",
     "--batch-size", "2",
     "--grad-accum", "8",
     "--learning-rate", "0.0001",
     "--max-seq-length", "1536"],
    check=True
)
print("✅ Training process exited successfully:", OUTPUT_DIR)


## Evaluate on held-out synthetic cases

This runs the fine-tuned adapter on held-out examples and (for comparison) the untouched base model. The report includes JSON parse rate, decision accuracy, issue-code precision/recall/F1, exact issue-set accuracy, and the rate at which flagged synthetic cases were incorrectly passed.

**Do not describe these synthetic metrics as ICAI grading accuracy.**


In [ ]:
EVAL_DIR = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_evaluation_v1_20261017"
subprocess.run(
    [sys.executable, "evaluate_supervisor.py",
     "--test-file", "/content/ca-evaluation-supervisor-data/test.jsonl",
     "--adapter-dir", OUTPUT_DIR,
     "--base-model", "HuggingFaceTB/SmolLM2-360M-Instruct",
     "--output-dir", EVAL_DIR,
     "--limit", "120",
     "--compare-base"],
    check=True
)
metrics_path = Path(EVAL_DIR) / "metrics.json"
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))


In [ ]:
# Package only the root adapter/tokenizer files and evaluation evidence.
# Trainer checkpoint directories are intentionally excluded from the deployment package.
import shutil, zipfile

EXPORT_DIR = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_smollm2_360m_v1_20261017_EXPORT"
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
adapter_root = Path(OUTPUT_DIR)
for file_path in adapter_root.iterdir():
    if file_path.is_file():
        shutil.copy2(file_path, Path(EXPORT_DIR) / file_path.name)
for name in ["metrics.json", "base_metrics.json", "predictions.jsonl", "base_predictions.jsonl"]:
    source = Path(EVAL_DIR) / name
    if source.is_file():
        shutil.copy2(source, Path(EXPORT_DIR) / name)

package_path = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_adapter_package_20261017.zip"
with zipfile.ZipFile(package_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in Path(EXPORT_DIR).iterdir():
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.name)

print("✅ Adapter package:", package_path)
print("✅ Evaluation evidence:", str(Path(EXPORT_DIR) / "metrics.json"))
print("⚠️ This is a research artifact; it is not wired into the live website yet.")


## Deployment gate — intentionally not automatic

The model has **not** been deployed by running this notebook. Next, review:
1. base vs fine-tuned synthetic metrics;
2. whether the adapter returns valid JSON and avoids false PASS decisions;
3. a separate benchmark of human-reviewed, legally permitted real evaluation examples.

Only then should we build an inference endpoint and connect it to the website. The model must never override the deterministic integrity gates or approve a copy when references, answer mapping, or handwriting are unresolved.
